In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("elikplim/eergy-efficiency-dataset")

print("Path to dataset files:", path)

In [ ]:
raw_data = open("/home/aditya0oo0/.cache/kagglehub/datasets/elikplim/eergy-efficiency-dataset/versions/1/ENB2012_data.csv", 'r').readlines()[1:]
visited = []
for i in raw_data:
    if i not in visited:
        visited.append(i)        ### No duplication in data
raw_data = visited
del visited
raw_data[:2]

In [ ]:
import numpy as np
import torch
# filter the data

data = np.array([i[:-1].split(',') for i in raw_data], dtype=np.float32)   # makes ndarray from str
# del raw_data

n = int(0.9*data.shape[0])

train_data = torch.tensor(data[:n], dtype=torch.float32)
test_data = torch.tensor(data[n:], dtype=torch.float32)

train_data.shape, test_data

In [ ]:
# inputs and targets split

inputs = train_data[:, :-2]      
targets = train_data[:, -2:]

test_inputs = test_data[:, :-2]
test_targets = test_data[:, -2:]
inputs, targets, test_inputs, test_targets

In [ ]:
inputs_norm = torch.hstack([inputs[:, :4] / 1000, inputs[:, 4:]/10])
out_norms = targets

test_inputs_norm = torch.hstack([test_inputs[:, :4]/1000, test_inputs[:, 4:]/10])
test_out_norms = test_targets

inputs_norm, out_norms, test_inputs_norm, test_out_norms

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline


In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.l_stack = nn.Sequential(
            nn.Linear(8, 512), nn.ReLU(),
            nn.Linear(512, 512), nn.ReLU(),
            nn.Linear(512, 2)
        )

    def forward(self, x):
        return self.l_stack(x)





In [ ]:
torch.tensor(3).item()

In [ ]:
model = NeuralNetwork()



def loss_fn0(pred, y):
    pred, y = abs(pred), abs(y)
    
    loss = abs(y - pred)      # loss tensor (x, 2) -> loss 1 item tensor
    
    return loss.sum()

    
    

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)


def eval_loss(model, loss_fxn):
    model.eval()
    loss = loss_fxn(model(test_inputs_norm), test_out_norms)
    model.train()
    return loss.item()

loss_fn = nn.L1Loss()


In [ ]:
# print(model(test_inputs).shape, test_targets.shape)
# loss_fn0(model(test_inputs), test_targets)
torch.rand((3, 4)).sum()

In [ ]:
# Training loop
EPOCH = 1000

y_loss = []
for i in range(EPOCH):
    model.train()

    optimizer.zero_grad(set_to_none=True)
    # Forward pass
    pred = model(inputs_norm)

    loss = loss_fn(pred, out_norms)

    # Backward pass
    loss.backward()

    # Update
    
    optimizer.step()


    if i % 200 == 0:

        # r_loss = eval_loss(model, loss_fn)
        print(f"Training Loss: {loss.item():.4f}")
        print(f"Eval loss: {eval_loss(model, loss_fn)}")
    y_loss.append(loss.item())




    

In [ ]:
plt.plot(y_loss)